# AI-Powered Personal Task Manager

**Assignment:** Build a GenAI-powered assistant that manages tasks through natural-language instructions.

This notebook implements a task manager using Python, Mistral AI, LangChain/LangGraph, MCP, SQLite and Streamlit. Users can create, list, search, retrieve, update, complete and delete tasks through plain English.

**Contents**
1. Problem understanding and proposed approach
2. Tech stack and justification
3. Python implementation (setup, database, MCP server, agent, Streamlit UI)
4. Sample inputs and outputs
5. Testing of core functionality and error handling
6. Limitations and future improvements

## 1. Problem understanding and proposed approach

People lose track of tasks and deadlines when managing them needs forms or exact commands. The goal is an assistant that understands requests such as:

- "Add a task to prepare for the GenAI assignment by tomorrow."
- "Show me all my pending tasks."
- "Mark the Python assignment as completed."
- "When is my Python assignment due?"

It must also handle **ambiguous requests** (missing title, several matching tasks) and **tasks that do not exist**.

### Proposed approach

1. The user types a natural-language request (notebook or Streamlit chat).
2. Mistral AI interprets the intent.
3. A LangGraph agent picks the right MCP tool.
4. The MCP server validates input and reads or writes SQLite.
5. The agent answers using only what the tool returned.

**Design principle:** the LLM only interprets intent. All validation (dates, categories, duplicates, missing IDs, title search) happens in deterministic Python inside the tools, and SQLite is the single source of truth. The assistant must never claim a change happened unless a tool confirms it.

## 2. Tech stack and justification

| Technology | Role | Why it is suitable |
|---|---|---|
| Python | Implementation language | Mandatory; mature AI and database libraries. |
| Mistral AI (`ministral-3b-2512`) | Intent understanding and replies | Small, fast and cheap; adequate because tools do the validation. |
| LangChain | Model interface | Consistent API for the chat model and tool calling. |
| LangGraph | Tool-using agent loop | Lets the model call tools, read results and answer. |
| MCP (FastMCP) | Tool protocol | Task operations are exposed as standard, reusable tools. |
| SQLite | Persistent storage | Local, zero-setup, file based. |
| Streamlit | Chat interface | Quick browser demo. |
| python-dotenv | Configuration | Keeps the API key out of the code. |

**Why MCP instead of plain function calling?** Plain function calling would also meet the assignment. MCP was chosen because the task tools run as a separate server that any MCP-compatible client can reuse, and the same server is used unchanged by both the notebook and the Streamlit app.

### Environment setup

```bash
pip install langchain-mistralai langgraph langchain langchain-mcp-adapters mcp python-dotenv streamlit pandas
```

Create a `.env` file in the project folder:

```text
MISTRAL_API_KEY=your_own_api_key_here
```

The `.env` file is **not** part of the submission.

## 3. Python implementation
### 3.1 Imports and environment

In [1]:
import os
import sys
import sqlite3
import asyncio
from datetime import datetime, timedelta
from pathlib import Path
from dotenv import load_dotenv

import pandas as pd
from langchain_mistralai import ChatMistralAI
from langchain_mcp_adapters.client import MultiServerMCPClient

load_dotenv()
MISTRAL_API_KEY = os.getenv("MISTRAL_API_KEY")
if not MISTRAL_API_KEY:
    raise ValueError("MISTRAL_API_KEY is missing. Add it to your .env file.")

PROJECT_DIR = Path.cwd()
DB_PATH = PROJECT_DIR / "tasks.db"
SERVER_PATH = PROJECT_DIR / "task_server.py"
print("Environment loaded.")

Environment loaded.


### 3.2 MCP server (tools + SQLite)

The server file creates its own database schema, so it works on its own. Tools:

| Tool | Purpose |
|---|---|
| `get_today` | Returns today's date so relative dates are resolved from the real clock. |
| `create_task` | Creates a task; rejects empty titles, bad or past dates, and duplicate pending titles. |
| `list_tasks` | Lists tasks, optionally filtered by status or category. |
| `find_tasks` | Searches titles by keyword (SQL `LIKE`) so matching is deterministic. |
| `get_task` | Full details of one task by ID. |
| `update_task` | Changes chosen fields; the value `CLEAR` empties an optional field. |
| `complete_task` | Marks a pending task completed. |
| `delete_task` | Deletes a task, only when `confirm=True`. |

In [2]:
server_code = r'''
import os
import sqlite3
from datetime import datetime, date
from pathlib import Path
try:
    from mcp.server.fastmcp import FastMCP          # mcp 1.x
except ImportError:
    from mcp.server.mcpserver import MCPServer as FastMCP   # mcp 2.x

DEFAULT_DB = Path(__file__).resolve().with_name("tasks.db")
CATEGORIES = ["Personal", "Work", "Study", "Finance", "Health", "Shopping", "Other"]
PRIORITIES = ["Low", "Medium", "High"]
CLEAR = "CLEAR"

mcp = FastMCP("Task Manager")


def db_path():
    return Path(os.environ.get("TASK_DB_PATH", DEFAULT_DB))


def get_connection():
    conn = sqlite3.connect(db_path())
    conn.row_factory = sqlite3.Row
    conn.execute("""
        CREATE TABLE IF NOT EXISTS tasks (
            ID INTEGER PRIMARY KEY AUTOINCREMENT,
            Title TEXT NOT NULL,
            Category TEXT NOT NULL DEFAULT 'Personal',
            Description TEXT NOT NULL DEFAULT '',
            Priority TEXT NOT NULL DEFAULT 'Medium'
                CHECK(Priority IN ('Low', 'Medium', 'High')),
            due_date TEXT NOT NULL DEFAULT '',
            Status TEXT NOT NULL DEFAULT 'Pending'
                CHECK(Status IN ('Pending', 'Completed')),
            Created_at TEXT NOT NULL,
            Updated_at TEXT NOT NULL,
            Completion_at TEXT
        )
    """)
    return conn


def current_timestamp():
    return datetime.now().astimezone().isoformat(timespec="seconds")


def check_date(value):
    """Return an error message, or None if value is a valid, non-past YYYY-MM-DD."""
    try:
        parsed = datetime.strptime(value, "%Y-%m-%d")
        if parsed.strftime("%Y-%m-%d") != value:
            raise ValueError
    except ValueError:
        return "Use a valid YYYY-MM-DD date."
    if parsed.date() < date.today():
        return "Due date is in the past."
    return None


@mcp.tool()
def get_today() -> dict:
    """Return today's date (YYYY-MM-DD) and weekday. Use it to resolve 'today', 'tomorrow', 'next Friday'."""
    today = date.today()
    return {"success": True, "today": today.isoformat(), "weekday": today.strftime("%A")}


@mcp.tool()
def create_task(title: str, category: str = "Personal", description: str = "",
                priority: str = "Medium", due_date: str = "",
                allow_duplicate: bool = False) -> dict:
    """Create a task. due_date must be YYYY-MM-DD or empty. A pending task with the
    same title is refused unless allow_duplicate is true."""
    if not isinstance(title, str) or not title.strip():
        return {"success": False, "message": "Title cannot be empty."}
    if category not in CATEGORIES:
        return {"success": False, "message": f"Invalid category. Choose from {CATEGORIES}."}
    if priority not in PRIORITIES:
        return {"success": False, "message": "Invalid priority. Use Low, Medium or High."}
    if due_date:
        error = check_date(due_date)
        if error:
            return {"success": False, "message": error}
    now = current_timestamp()
    with get_connection() as conn:
        if not allow_duplicate:
            dup = conn.execute(
                "SELECT ID FROM tasks WHERE lower(Title) = lower(?) AND Status = 'Pending'",
                (title.strip(),)).fetchone()
            if dup:
                return {"success": False, "duplicate_of": dup["ID"],
                        "message": f"A pending task with this title already exists (ID {dup['ID']}). "
                                   "Ask the user whether to create it anyway."}
        cursor = conn.execute(
            """INSERT INTO tasks (Title, Category, Description, Priority, due_date,
               Status, Created_at, Updated_at, Completion_at)
               VALUES (?, ?, ?, ?, ?, 'Pending', ?, ?, NULL)""",
            (title.strip(), category, description, priority, due_date, now, now))
        task_id = cursor.lastrowid
    return {"success": True, "task_id": task_id, "title": title.strip(),
            "category": category, "priority": priority, "due_date": due_date,
            "status": "Pending", "created_at": now}


@mcp.tool()
def list_tasks(status: str = "", category: str = "") -> dict:
    """List tasks; optionally filter by status (Pending/Completed) and category."""
    query, params = "SELECT * FROM tasks WHERE 1=1", []
    if status:
        if status not in ["Pending", "Completed"]:
            return {"success": False, "message": "Status must be Pending or Completed."}
        query += " AND Status = ?"
        params.append(status)
    if category:
        query += " AND Category = ?"
        params.append(category)
    query += " ORDER BY ID DESC"
    with get_connection() as conn:
        rows = conn.execute(query, params).fetchall()
    return {"success": True, "count": len(rows), "tasks": [dict(r) for r in rows]}


@mcp.tool()
def find_tasks(keyword: str, status: str = "") -> dict:
    """Find tasks whose title contains the keyword (case-insensitive). Use this to
    identify a task by name before completing, updating, deleting or describing it.
    If count > 1 the request is ambiguous: ask the user which task they mean."""
    if not isinstance(keyword, str) or not keyword.strip():
        return {"success": False, "message": "Keyword cannot be empty."}
    query = "SELECT * FROM tasks WHERE lower(Title) LIKE ?"
    params = [f"%{keyword.strip().lower()}%"]
    if status:
        if status not in ["Pending", "Completed"]:
            return {"success": False, "message": "Status must be Pending or Completed."}
        query += " AND Status = ?"
        params.append(status)
    with get_connection() as conn:
        rows = conn.execute(query + " ORDER BY ID DESC", params).fetchall()
    return {"success": True, "count": len(rows), "tasks": [dict(r) for r in rows]}


@mcp.tool()
def get_task(task_id: int) -> dict:
    """Get full details (due date, status, etc.) for one task by ID."""
    with get_connection() as conn:
        row = conn.execute("SELECT * FROM tasks WHERE ID = ?", (task_id,)).fetchone()
    if row is None:
        return {"success": False, "message": f"Task {task_id} not found."}
    return {"success": True, "task": dict(row)}


@mcp.tool()
def update_task(task_id: int, title: str = "", category: str = "",
                description: str = "", priority: str = "", due_date: str = "") -> dict:
    """Update the supplied fields. Empty string = leave unchanged.
    Pass CLEAR as description or due_date to empty that field."""
    updates = {}
    if title:
        if not title.strip():
            return {"success": False, "message": "Title cannot be empty."}
        updates["Title"] = title.strip()
    if category:
        if category not in CATEGORIES:
            return {"success": False, "message": f"Invalid category. Choose from {CATEGORIES}."}
        updates["Category"] = category
    if description:
        updates["Description"] = "" if description == CLEAR else description
    if priority:
        if priority not in PRIORITIES:
            return {"success": False, "message": "Invalid priority. Use Low, Medium or High."}
        updates["Priority"] = priority
    if due_date:
        if due_date == CLEAR:
            updates["due_date"] = ""
        else:
            error = check_date(due_date)
            if error:
                return {"success": False, "message": error}
            updates["due_date"] = due_date
    if not updates:
        return {"success": False, "message": "No updates provided."}
    now = current_timestamp()
    with get_connection() as conn:
        if conn.execute("SELECT ID FROM tasks WHERE ID = ?", (task_id,)).fetchone() is None:
            return {"success": False, "message": f"Task {task_id} not found."}
        assignments = ", ".join(f"{field} = ?" for field in updates)
        conn.execute(f"UPDATE tasks SET {assignments}, Updated_at = ? WHERE ID = ?",
                     (*updates.values(), now, task_id))
    return {"success": True, "task_id": task_id, "updated_fields": list(updates),
            "message": "Task updated."}


@mcp.tool()
def complete_task(task_id: int) -> dict:
    """Mark a pending task as completed and store the completion time."""
    now = current_timestamp()
    with get_connection() as conn:
        row = conn.execute("SELECT Title, Status FROM tasks WHERE ID = ?", (task_id,)).fetchone()
        if row is None:
            return {"success": False, "message": f"Task {task_id} not found."}
        if row["Status"] == "Completed":
            return {"success": False, "message": "Task is already completed."}
        conn.execute("""UPDATE tasks SET Status = 'Completed', Completion_at = ?,
                     Updated_at = ? WHERE ID = ?""", (now, now, task_id))
    return {"success": True, "task_id": task_id, "title": row["Title"], "completion_at": now}


@mcp.tool()
def delete_task(task_id: int, confirm: bool = False) -> dict:
    """Permanently delete a task. Only call with confirm=true after the user
    explicitly asked to delete it."""
    if not confirm:
        return {"success": False, "message": "Deletion not confirmed. Ask the user to confirm."}
    with get_connection() as conn:
        row = conn.execute("SELECT Title FROM tasks WHERE ID = ?", (task_id,)).fetchone()
        if row is None:
            return {"success": False, "message": f"Task {task_id} not found."}
        conn.execute("DELETE FROM tasks WHERE ID = ?", (task_id,))
    return {"success": True, "task_id": task_id, "title": row["Title"], "message": "Task deleted."}


if __name__ == "__main__":
    mcp.run(transport="stdio")
'''
SERVER_PATH.write_text(server_code, encoding="utf-8")
print("MCP server written to:", SERVER_PATH.name)

MCP server written to: task_server.py


### 3.3 Connect Mistral, LangGraph and MCP

The system prompt is rebuilt for every request, so "today" is never stale. The agent also has a `get_today` tool.

In [3]:
try:
    from langchain.agents import create_agent          # current LangChain/LangGraph v1 API
    def build_agent(model, tools, prompt):
        return create_agent(model, tools, system_prompt=prompt)
except ImportError:                                     # older versions
    from langgraph.prebuilt import create_react_agent
    def build_agent(model, tools, prompt):
        return create_react_agent(model, tools, prompt=prompt)

model = ChatMistralAI(model="ministral-3b-2512", api_key=MISTRAL_API_KEY, temperature=0)

client = MultiServerMCPClient({
    "task_manager": {"transport": "stdio", "command": sys.executable, "args": [str(SERVER_PATH)]}
})


def system_prompt():
    today = datetime.now().astimezone().date()
    return (
        "You are a personal task manager. Today is "
        f"{today.isoformat()} ({today.strftime('%A')}).\n"
        "RULES:\n"
        "- Use tools for every task operation. Never invent tasks, IDs or results.\n"
        "- Never claim success unless a tool returned success=true.\n"
        "- Convert relative dates (tomorrow, next Friday) to YYYY-MM-DD; call get_today if unsure.\n"
        "- To act on a task named by the user, call find_tasks first. If it returns 0 matches, say the "
        "task does not exist. If it returns more than one, list them with IDs and ask which one. "
        "Do not guess.\n"
        "- If the title is missing or unclear when creating a task, ask for it. Do not invent a title.\n"
        "- If create_task reports a duplicate, tell the user and ask whether to create it anyway.\n"
        "- Infer category and priority when clear; otherwise use defaults.\n"
        "- Delete only when the user explicitly asks, and pass confirm=true only then.\n"
        "- Be concise. For lists include ID, title, category, priority, due date and status."
    )


async def load_agent():
    tools = await client.get_tools()
    return tools, build_agent(model, tools, system_prompt())


async def ask_task_manager(message, history=None):
    """Ask the agent one question. A fresh agent is built each call so the date is current."""
    tools = await client.get_tools()
    agent = build_agent(model, tools, system_prompt())
    messages = (history or []) + [{"role": "user", "content": message}]
    result = await agent.ainvoke({"messages": messages}, config={"recursion_limit": 20})
    content = result["messages"][-1].content
    if isinstance(content, list):
        content = "\n".join(b.get("text", "") for b in content if isinstance(b, dict))
    return content.strip()


tools, _ = await load_agent()
print("MCP tools loaded:", [t.name for t in tools])

MCP tools loaded: ['get_today', 'create_task', 'list_tasks', 'find_tasks', 'get_task', 'update_task', 'complete_task', 'delete_task']


### 3.4 Streamlit user interface

In [4]:
streamlit_code = r'''
import os, sys, asyncio, logging
from pathlib import Path
from datetime import datetime

import streamlit as st
from dotenv import load_dotenv
from langchain_mistralai import ChatMistralAI
from langchain_mcp_adapters.client import MultiServerMCPClient

try:
    from langchain.agents import create_agent
    def build_agent(model, tools, prompt):
        return create_agent(model, tools, system_prompt=prompt)
except ImportError:
    from langgraph.prebuilt import create_react_agent
    def build_agent(model, tools, prompt):
        return create_react_agent(model, tools, prompt=prompt)

st.set_page_config(page_title="AI Task Manager", page_icon="✅", layout="centered")
logging.basicConfig(level=logging.INFO)

BASE_DIR = Path(__file__).resolve().parent
load_dotenv(BASE_DIR / ".env")
API_KEY = os.getenv("MISTRAL_API_KEY")
SERVER_PATH = BASE_DIR / "task_server.py"
MODEL_NAME = "ministral-3b-2512"

if not API_KEY:
    st.error("MISTRAL_API_KEY is missing from your .env file.")
    st.stop()
if not SERVER_PATH.is_file():
    st.error(f"MCP server not found: {SERVER_PATH.name}")
    st.stop()


@st.cache_resource
def get_model():
    return ChatMistralAI(model=MODEL_NAME, api_key=API_KEY, temperature=0)


def system_prompt():
    today = datetime.now().astimezone().date()
    return (
        f"You are a personal task manager. Today is {today.isoformat()} ({today.strftime('%A')}).\n"
        "RULES:\n"
        "- Use tools for every task operation. Never invent tasks, IDs or results.\n"
        "- Never claim success unless a tool returned success=true.\n"
        "- Convert relative dates to YYYY-MM-DD; call get_today if unsure.\n"
        "- To act on a task named by the user, call find_tasks first. 0 matches: say it does not exist. "
        "More than one: list them with IDs and ask which one. Do not guess.\n"
        "- If the title is missing when creating a task, ask for it. Do not invent one.\n"
        "- If create_task reports a duplicate, tell the user and ask whether to create it anyway.\n"
        "- If you asked for a task ID and the user replies with a number, continue the original operation.\n"
        "- Delete only when explicitly asked, and pass confirm=true only then.\n"
        "- Be concise. For lists include ID, title, category, priority, due date and status."
    )


async def run_agent(messages):
    client = MultiServerMCPClient({"task_manager": {
        "transport": "stdio", "command": sys.executable, "args": [str(SERVER_PATH)]}})
    tools = await client.get_tools()
    agent = build_agent(get_model(), tools, system_prompt())
    result = await agent.ainvoke({"messages": messages}, config={"recursion_limit": 20})
    content = result["messages"][-1].content
    if isinstance(content, list):
        content = "\n".join(b.get("text", "") for b in content if isinstance(b, dict))
    answer = content.strip()
    if not answer:
        raise RuntimeError("The agent returned an empty response.")
    return answer


st.title("AI-Powered Personal Task Manager")
st.caption("Manage your tasks using natural language.")

with st.sidebar:
    st.subheader("Task Manager")
    st.caption(f"Model: {MODEL_NAME}")
    st.caption(f"Today: {datetime.now().astimezone().date()}")
    if st.button("Clear chat history", use_container_width=True):
        st.session_state.messages = []
        st.rerun()

if "messages" not in st.session_state:
    st.session_state.messages = []

for m in st.session_state.messages:
    with st.chat_message(m["role"]):
        st.markdown(m["content"])

user_input = st.chat_input("e.g. Add a high-priority Study task due tomorrow")
if user_input:
    st.session_state.messages.append({"role": "user", "content": user_input})
    with st.chat_message("user"):
        st.markdown(user_input)
    with st.chat_message("assistant"):
        with st.spinner("Working on your tasks..."):
            try:
                answer = asyncio.run(run_agent(st.session_state.messages))
            except Exception:
                logging.exception("Agent execution failed.")
                answer = "The request failed. Check the terminal logs for the underlying error."
            st.markdown(answer)
    st.session_state.messages.append({"role": "assistant", "content": answer})
'''
app_path = PROJECT_DIR / "app.py"
app_path.write_text(streamlit_code, encoding="utf-8")
compile(streamlit_code, "app.py", "exec")      # syntax check
print("app.py written and syntax-checked. Run it with:  streamlit run app.py")

app.py written and syntax-checked. Run it with:  streamlit run app.py


## 4. Sample inputs and outputs (live agent)

Each cell sends one natural-language request to the Mistral agent. The tasks used here are clearly named demo tasks. Run the cells in order; the outputs shown are whatever your run produced.

### 4.1 Create a task

In [5]:
print(await ask_task_manager("Add a task to prepare for the GenAI assignment by tomorrow."))

Task created: **ID 1**, *Prepare for GenAI assignment* (Study, High, Due 2026-10-22, Pending).


### 4.2 Create a task with category and priority

In [6]:
print(await ask_task_manager("Add a high priority study task called Python assignment due in 3 days."))

Added task **Python assignment** (ID: 2) to **Study** category, **High** priority, due **2026-10-24**.


### 4.3 View pending tasks

In [7]:
print(await ask_task_manager("Show me all my pending tasks."))

Here are your pending tasks:

1. **ID: 1**
   - Title: Prepare for GenAI assignment
   - Category: Study
   - Priority: High
   - Due: 2026-10-22
   - Status: Pending

2. **ID: 2**
   - Title: Python assignment
   - Category: Study
   - Priority: High
   - Due: 2026-10-24
   - Status: Pending


### 4.4 Retrieve task details

In [8]:
print(await ask_task_manager("When is my Python assignment due, and what is its status?"))

Your Python assignment is due on **2026-10-24** (Thursday) and its status is **Pending**.


### 4.5 Mark a task as completed

In [9]:
print(await ask_task_manager("Mark the Python assignment as completed."))

Python assignment marked as completed.


### 4.6 View completed tasks

In [10]:
print(await ask_task_manager("Show me my completed tasks."))

Here’s your completed task:

- **ID:** 2
  - **Title:** Python assignment
  - **Category:** Study
  - **Priority:** High
  - **Due Date:** 2026-10-24
  - **Status:** Completed


## 5. Testing core functionality and error handling

### 5.1 Tool-level tests (deterministic, no LLM)

These call the MCP tool functions directly on a **temporary database**, so they do not touch your real tasks and do not depend on the model. Each `assert` fails loudly if behaviour is wrong.

In [11]:
import tempfile, importlib

test_dir = tempfile.mkdtemp()
os.environ["TASK_DB_PATH"] = os.path.join(test_dir, "test_tasks.db")
sys.path.insert(0, str(PROJECT_DIR))
import task_server
importlib.reload(task_server)
T = task_server

today = datetime.now().astimezone().date()
tomorrow = (today + timedelta(days=1)).isoformat()
yesterday = (today - timedelta(days=1)).isoformat()
results = []

def check(name, condition, detail=""):
    results.append({"Test": name, "Result": "PASS" if condition else "FAIL", "Detail": detail})
    assert condition, f"{name} failed: {detail}"

# 1. Create
r = T.create_task("Prepare GenAI assignment", due_date=tomorrow)
check("Create task with due date", r["success"] and r["status"] == "Pending", str(r["task_id"]))
tid = r["task_id"]

# 2. Retrieve details
r = T.get_task(tid)
check("Retrieve due date and status", r["success"] and r["task"]["due_date"] == tomorrow
      and r["task"]["Status"] == "Pending", f"due={r['task']['due_date']}, status={r['task']['Status']}")

# 3. List pending
r = T.list_tasks(status="Pending")
check("List pending tasks", r["success"] and r["count"] == 1)

# 4. Complete
r = T.complete_task(tid)
check("Complete task", r["success"] and r["completion_at"])
t = T.get_task(tid)["task"]
check("Stored status is Completed", t["Status"] == "Completed" and t["Completion_at"])
check("Completed task leaves pending list", T.list_tasks(status="Pending")["count"] == 0)

# 5. Completing twice
check("Complete already-completed task", not T.complete_task(tid)["success"])

# 6. Missing task
check("Get non-existent task (999999)", "not found" in T.get_task(999999)["message"])
check("Complete non-existent task (999999)", "not found" in T.complete_task(999999)["message"])
check("Update non-existent task (999999)", "not found" in T.update_task(999999, title="x")["message"])
check("Delete non-existent task (999999)", "not found" in T.delete_task(999999, confirm=True)["message"])

# 7. Validation
check("Reject empty title", not T.create_task("   ")["success"])
check("Reject invalid date 2026-02-30", not T.create_task("Bad date", due_date="2026-02-30")["success"])
check("Reject wrong date format", not T.create_task("Bad format", due_date="25/10/2026")["success"])
check("Reject past due date", "past" in T.create_task("Old task", due_date=yesterday)["message"])
check("Reject invalid priority", not T.create_task("P", priority="Urgent")["success"])
check("Reject invalid category", not T.create_task("C", category="Hobby")["success"])

# 8. Ambiguity: two similar tasks
a = T.create_task("Python assignment", due_date=tomorrow)["task_id"]
b = T.create_task("Python assignment part 2", due_date=tomorrow)["task_id"]
r = T.find_tasks("python assignment")
check("Ambiguous title returns multiple matches", r["count"] == 2, f"ids={[x['ID'] for x in r['tasks']]}")
check("Unknown title returns zero matches", T.find_tasks("zzz-nothing")["count"] == 0)
check("Empty search keyword rejected", not T.find_tasks("")["success"])

# 9. Duplicate protection
r = T.create_task("Python assignment", due_date=tomorrow)
check("Duplicate pending title refused", not r["success"] and r["duplicate_of"] == a)
check("Duplicate allowed when confirmed", T.create_task("Python assignment", allow_duplicate=True)["success"])

# 10. Update, including clearing a field
check("Update priority", T.update_task(a, priority="High")["success"])
check("Priority saved", T.get_task(a)["task"]["Priority"] == "High")
check("Clear due date", T.update_task(a, due_date="CLEAR")["success"] and T.get_task(a)["task"]["due_date"] == "")
check("Update with nothing supplied rejected", not T.update_task(a)["success"])

# 11. Delete safety
check("Delete refused without confirm", not T.delete_task(b)["success"] and T.get_task(b)["success"])
check("Delete with confirm", T.delete_task(b, confirm=True)["success"] and not T.get_task(b)["success"])

# 12. Persistence: a brand-new connection still sees the data
import sqlite3 as _sq
with _sq.connect(os.environ["TASK_DB_PATH"]) as c:
    n = c.execute("SELECT COUNT(*) FROM tasks").fetchone()[0]
check("Persistence across new connection", n >= 3, f"{n} rows on disk")

pd.DataFrame(results)

,Test,Result,Detail
0,Create task with due date,PASS,1
1,Retrieve due date and status,PASS,"due=2026-10-22, status=Pending"
2,List pending tasks,PASS,
3,Complete task,PASS,
4,Stored status is Completed,PASS,
5,Completed task leaves pending list,PASS,
6,Complete already-completed task,PASS,
7,Get non-existent task (999999),PASS,
8,Complete non-existent task (999999),PASS,
9,Update non-existent task (999999),PASS,


### 5.2 Agent-level tests: ambiguity and missing tasks

These need the live model, so they go through the agent. They point at the **real** `tasks.db`. First, switch back from the temporary test database.

In [12]:
os.environ.pop("TASK_DB_PATH", None)
print("Using real database:", DB_PATH.name)

Using real database: tasks.db


**Missing task**

In [13]:
print(await ask_task_manager("Mark task 999999 as completed."))

[10/21/26 18:07:09] INFO     HTTP Request: POST https://api.mistral.ai/v1/chat/completions          ]8;id=4032195;file:///Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=4032196;file:///Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/httpx/_client.py#1740\1740]8;;\
                             "HTTP/1.1 200 OK"                                                                     

[10/21/26 18:07:11] INFO     HTTP Request: POST https://api.mistral.ai/v1/chat/completions          ]8;id=4032201;file:///Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=4032202;file:///Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/httpx/_client.py#1740\1740]8;;\
                             "HTTP/1.1 200 OK"                                                                     

Task 999999 does not exist. Please check the ID or ask for clarification.


**Non-existent task by name**

In [14]:
print(await ask_task_manager("Mark the Quantum Physics homework as completed."))

[10/21/26 18:07:13] INFO     HTTP Request: POST https://api.mistral.ai/v1/chat/completions          ]8;id=4032207;file:///Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=4032208;file:///Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/httpx/_client.py#1740\1740]8;;\
                             "HTTP/1.1 200 OK"                                                                     

[10/21/26 18:07:14] INFO     HTTP Request: POST https://api.mistral.ai/v1/chat/completions          ]8;id=4032213;file:///Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=4032214;file:///Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/httpx/_client.py#1740\1740]8;;\
                             "HTTP/1.1 200 OK"                                                                     

The task **"Quantum Physics homework"** does not exist. Did you mean something else?


**Missing title (ambiguous request)**

In [15]:
print(await ask_task_manager("Add a task."))

[10/21/26 18:07:17] INFO     HTTP Request: POST https://api.mistral.ai/v1/chat/completions          ]8;id=4032219;file:///Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=4032220;file:///Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/httpx/_client.py#1740\1740]8;;\
                             "HTTP/1.1 200 OK"                                                                     

What would you like to name the task? And provide a brief description if needed.


**Invalid date**

In [16]:
print(await ask_task_manager("Add a task called Pay fees due 2026-02-30."))

[10/21/26 18:07:21] INFO     HTTP Request: POST https://api.mistral.ai/v1/chat/completions          ]8;id=4032225;file:///Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=4032226;file:///Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/httpx/_client.py#1740\1740]8;;\
                             "HTTP/1.1 200 OK"                                                                     

[10/21/26 18:07:23] INFO     HTTP Request: POST https://api.mistral.ai/v1/chat/completions          ]8;id=4032231;file:///Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=4032232;file:///Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/httpx/_client.py#1740\1740]8;;\
                             "HTTP/1.1 200 OK"                                                                     

The date **2026-02-30** is invalid (February has only 28 or 29 days). Please use a valid date, e.g., **2026-03-31** or **2026-04-01**.


**Ambiguous title.** First create two similar tasks, then try to complete one by a shared word.

In [17]:
print(await ask_task_manager("Add a task called Project report draft due tomorrow."))
print("-----")
print(await ask_task_manager("Add a task called Project report review due tomorrow."))
print("-----")
print(await ask_task_manager("Mark the project report as completed."))

[10/21/26 18:07:25] INFO     HTTP Request: POST https://api.mistral.ai/v1/chat/completions          ]8;id=4032237;file:///Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=4032238;file:///Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/httpx/_client.py#1740\1740]8;;\
                             "HTTP/1.1 200 OK"                                                                     

[10/21/26 18:07:27] INFO     HTTP Request: POST https://api.mistral.ai/v1/chat/completions          ]8;id=4032243;file:///Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=4032244;file:///Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/httpx/_client.py#1740\1740]8;;\
                             "HTTP/1.1 200 OK"                                                                     

[10/21/26 18:07:29] INFO     HTTP Request: POST https://api.mistral.ai/v1/chat/completions          ]8;id=4032249;file:///Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=4032250;file:///Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/httpx/_client.py#1740\1740]8;;\
                             "HTTP/1.1 200 OK"                                                                     

Added task **Project report draft** (ID: 3) due **2026-10-22**.
-----


[10/21/26 18:07:31] INFO     HTTP Request: POST https://api.mistral.ai/v1/chat/completions          ]8;id=4032255;file:///Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=4032256;file:///Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/httpx/_client.py#1740\1740]8;;\
                             "HTTP/1.1 200 OK"                                                                     

...

[10/21/26 18:07:34] INFO     HTTP Request: POST https://api.mistral.ai/v1/chat/completions          ]8;id=4032267;file:///Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=4032268;file:///Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/httpx/_client.py#1740\1740]8;;\
                             "HTTP/1.1 200 OK"                                                                     

Added task **Project report review** (ID: 4) due **2026-10-22** (tomorrow).
-----


[10/21/26 18:07:36] INFO     HTTP Request: POST https://api.mistral.ai/v1/chat/completions          ]8;id=4032273;file:///Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=4032274;file:///Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/httpx/_client.py#1740\1740]8;;\
                             "HTTP/1.1 200 OK"                                                                     

[10/21/26 18:07:38] INFO     HTTP Request: POST https://api.mistral.ai/v1/chat/completions          ]8;id=4032279;file:///Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=4032280;file:///Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/httpx/_client.py#1740\1740]8;;\
                             "HTTP/1.1 200 OK"                                                                     

The tasks matching "project report" are:
- **ID 4**: Project report review (Work, High, due 2026-10-22)
- **ID 3**: Project report draft (Work, High, due 2026-10-22)

Which one should I mark as completed?


**Persistence.** Re-open the database in a fresh connection, as would happen after restarting the app.

In [18]:
with sqlite3.connect(DB_PATH) as conn:
    conn.row_factory = sqlite3.Row
    rows = conn.execute("SELECT ID, Title, Category, Priority, due_date, Status, Completion_at "
                        "FROM tasks ORDER BY ID DESC").fetchall()
pd.DataFrame([dict(r) for r in rows])

,ID,Title,Category,Priority,due_date,Status,Completion_at
0,4,Project report review,Work,High,2026-10-22,Pending,NaN
1,3,Project report draft,Work,High,2026-10-22,Pending,NaN
2,2,Python assignment,Study,High,2026-10-24,Completed,2026-10-21T18:07:02+05:30
3,1,Prepare for GenAI assignment,Study,High,2026-10-22,Pending,NaN


### 5.3 Test summary

| Area | Covered by |
|---|---|
| Create with title and due date | 4.1, 4.2, 5.1 |
| View pending tasks | 4.3, 5.1 |
| Mark completed | 4.5, 5.1 |
| Retrieve due date and status | 4.4, 5.1 |
| Task that does not exist | 5.1, 5.2 |
| Ambiguous request (missing title, similar titles) | 5.1, 5.2 |
| Invalid input (date, priority, category, empty title) | 5.1, 5.2 |
| Persistence | 5.1, 5.2 |

Tool-level behaviour (5.1) is deterministic. Agent-level behaviour (4 and 5.2) depends on the language model, so read those outputs and confirm them against the database table above.

## 6. Limitations and future improvements

### Limitations
- A 3B-parameter model was chosen for speed and cost. It can misread unusual phrasing, so validation lives in the tools instead of the prompt.
- Relative dates are resolved by the model from the supplied date. The `get_today` tool and server-side date validation reduce, but do not remove, this risk.
- Title search is keyword based; it does not understand synonyms or typos.
- SQLite is a single-file local database: no multi-user access or device sync.
- The Streamlit app starts a fresh MCP server process for each message. This is simple and robust but slower than reusing one.
- Streamlit chat history is held in the browser session only; tasks themselves are stored permanently in SQLite.
- Agent-level tests were run by hand, not as an automated suite, and no accuracy or latency measurements were taken.

### Future improvements
- Automated agent evaluation (task-selection accuracy, date accuracy, latency).
- Reminders, recurring tasks, sorting and richer search (fuzzy matching).
- Reuse one MCP connection in Streamlit.
- Authentication and a server database for multi-user use.

## 7. How to run the Streamlit application

1. Run this notebook from top to bottom to create `task_server.py` and `app.py`.
2. Make sure `.env` contains your own `MISTRAL_API_KEY`.
3. In a terminal in the project folder run:

```bash
streamlit run app.py
```

4. Open the printed local URL and try the prompts from section 4.